In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.18.0


In [2]:
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

FOLDER_PATH = '/content/drive/MyDrive/Colab Notebooks/data_model_davies'

Mounted at /content/drive


In [3]:
# Verificar si se detecta la TPU
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # Detecta la TPU automáticamente
    print('TPU encontrada:', tpu.master())
except ValueError:
    print('No se encontró ninguna TPU.')

TPU encontrada: 


In [4]:
from sklearn.preprocessing import MinMaxScaler
from scipy.signal import find_peaks

def run_simulation(dij_sim, Ii, Zj, beta_r, gamma_r, alpha_p, gamma_p):
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    Zj_t = tf.convert_to_tensor(Zj, dtype=tf.float32)
    Ii_t = tf.convert_to_tensor(Ii, dtype=tf.float32)
    dij_t = tf.convert_to_tensor(dij_sim, dtype=tf.float32)

    Lr, Lp = 6, 12
    eta, tau, Ptotal = 0.005, 0.75, 500.0
    counter, idxr, idxp, t, dt = 0, 0, 0, 0, 0.1
    Nt, Ntt, nlat, nlon, nz = 500, 10, 78, 71, 500

    Rj_t = tf.zeros(nz, dtype=tf.float32)
    Ai_t = tf.zeros((nlat, nlon), dtype=tf.float32)
    Pj_t = tf.zeros(nz, dtype=tf.float32)
    Ci_t = tf.zeros((nlat, nlon), dtype=tf.float32)
    fjdel_t = tf.zeros((nz, Lr), dtype=tf.float32)
    rho_t = tf.ones((nlat, nlon), dtype=tf.float32)
    Ddel_t = tf.zeros((nz, Lp), dtype=tf.float32)
    Dj_t = tf.zeros(nz, dtype=tf.float32)

    auxij1 = tf.exp(-beta_r * dij_t)
    dij_t = 1.0 * Zj_t[:, 2] * auxij1 / tf.reduce_max(Zj_t[:, 2])

    print("Starting loop ...")
    timer0 = time.time()

    for nn in range(Nt):
        for mm in range(Ntt):
            fj_t = tf.exp(-tf.floor(gamma_r * Pj_t / (Rj_t + 1.0e-20))) # 500 Values
            Wij_t = fj_t * dij_t # Dj shape: 78, 71, 500 (Broadcasting) --> Attractiveness term ij
            Wi_t = tf.reduce_sum(Wij_t, axis=2) # Wi shape: 78, 71 each value is the sum of 500 values (fj) --> Attractiveness term i
            P_off_t = rho_t * Wi_t / (1.0 + Wi_t) # P_off shape: 78, 71
            idxr = counter - (Lr) * int(counter / Lr) # Lr: 6, idxr: 0-5
            # Use tf.tensor_scatter_nd_update to update fjdel_t
            indices = tf.range(0, nz)[:, tf.newaxis]
            indices = tf.concat([indices, tf.fill([nz, 1], idxr)], axis=1)
            fjdel_t = tf.tensor_scatter_nd_update(fjdel_t, indices, fj_t)
            dnm = Lr if counter >= Lr else counter + 1
            We_ij_t = tf.reduce_sum(fjdel_t, axis=1) * dij_t / dnm # Delayed term computation
            auxw = Ai_t / (tf.reduce_sum(We_ij_t, axis=2) + 1.0e-20)  # Flow computation step 1i
            Sij_t = auxw[:, :, tf.newaxis] * We_ij_t # Flow computation step 2
            Rj_t = tf.reduce_sum(tf.reduce_sum(Sij_t, axis=1), axis=0) # Rioter computation
            indices = tf.range(0, nz)
            updates = Zj_t[:, 2] ** (alpha_p) * tf.exp(gamma_p * Rj_t[:])
            Dj_t = tf.tensor_scatter_nd_update(Dj_t, indices[:, tf.newaxis], updates)

            idxp = counter - (Lp) * int(counter / Lp) # Delayed term computation
            # Use tf.tensor_scatter_nd_update to update Ddel_t
            indices = tf.range(nz)[:, None]
            indices = tf.concat([indices, tf.fill([nz, 1], idxp)], axis=1)
            Ddel_t = tf.tensor_scatter_nd_update(Ddel_t, indices, Dj_t) # Use tf.tensor_scatter_nd_update for assignment

            dnm = Lp if counter >= Lp else counter + 1
            Dej_t = tf.reduce_sum(Ddel_t, axis=1) / dnm
            Pj_t = Ptotal * Dej_t / tf.reduce_sum(Dej_t)
            counter += 1
            fj_t = 1.0 - tf.exp(-tf.floor(Pj_t / (Rj_t + 1.0e-20))) # Capture rate
            Ci_t = tau * tf.reduce_sum(Sij_t * fj_t, axis=2)
            Ai_t += dt * (eta * P_off_t * Ii_t - Ci_t) # Time step for Ai and Ii
            Ii_t += -dt * eta * P_off_t * Ii_t
            t += dt

    Rj_t = Rj_t.numpy()

    x = np.arange(len(Rj_t))
    y = Rj_t

    # Detectar picos
    y_norm = (y - np.min(y)) / (np.max(y) - np.min(y))  # Normalización Min-Max
    peaks, _ = find_peaks(y_norm, prominence=0.05)  # Ajuste según la escala normalizada
    peaks = np.insert(peaks, 0, 0)  # Asegurar que el primer punto sea considerado un pico
    peaks = np.append(peaks, len(y)-1)  # Incluir el último punto también si es necesario

    # Filtrar los valores
    x_peaks = x[peaks]
    y_peaks = y[peaks]

    # Interpolación lineal
    linear_interpolation_Rj = np.interp(x, x_peaks, y_peaks)

    scaler = MinMaxScaler()
    Rj_t = scaler.fit_transform(linear_interpolation_Rj.reshape(-1, 1)).flatten()

    timer1 = time.time()
    print("Total execution time: ", timer1 - timer0)

    return Rj_t

In [ ]:
# Initial data arrays
origin = np.loadtxt(f"{FOLDER_PATH}/origin_dens_500m_5am10am.dat") # Origin
destination = np.loadtxt(f"{FOLDER_PATH}/destination_dens_500m_5am10am.dat") # Destination

# Population, Police and Rioters
Ii = origin + destination  # Inactive population (the number of inactive individuals resident in area i)
Zj = np.loadtxt(f"{FOLDER_PATH}/targets_500.dat")  # Targets (Benefit for j site is given by the logarithm of Zj a non-dimensional measure of its relative value)
dij = np.load(f"{FOLDER_PATH}/rij_500_no_network.npy")

# PSO parameters
beta_r = 0.1
gamma_r = 0.19482765
alpha_p = 0.97356374
gamma_p = 0.03415131

Rj_simulated = run_simulation(dij, Ii, Zj, beta_r, gamma_r, alpha_p, gamma_p)

Starting loop ...


In [ ]:
Zj_t = tf.convert_to_tensor(Zj, dtype=tf.float32)